In [0]:
########################
#Install Dependencies
########################
%pip install openpyxl
%pip install pycountry
# dbutils.library.restartPython()

In [0]:
########################
#Setup for Transformation Tests
########################
this_notebook_state = "ended"

from datetime import datetime, timedelta
run_user = spark.sql("SELECT current_user()").first()[0]
run_tag = "Testing Transformation Tests"
run_by_automation_name = "Transformation_Tests"
run_start_datetime = datetime.now()

dbutils.widgets.text("state_under_test", "")
dbutils.widgets.text("fields_to_exclude", "")

state_under_test = dbutils.widgets.get("state_under_test")
fields_to_exclude = dbutils.widgets.get("fields_to_exclude")

if state_under_test == "":
    state_under_test = this_notebook_state

if fields_to_exclude != "":
    fields_to_exclude = [item.strip() for item in fields_to_exclude.split(",")]
else:
    fields_to_exclude = []

child_fields_to_exclude = []
fields_to_exclude.extend(child_fields_to_exclude)

print(f"Fields to exclude = {str(fields_to_exclude)}")
print(f"Testing State = {state_under_test}")


from models.test_result import TestResult
from dataclasses import asdict
import os

all_test_results = []
current_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
base_path = current_path.rsplit("/", 1)[0] + "/"
test_results_path = "/Workspace/Users/" + run_user + "/Results/Transformation_Tests"
os.makedirs(test_results_path, exist_ok=True)


In [0]:
#Load Config and Setup Enviorment Variables
M2_bronze = None
M3_bronze = None
M3_silver = None
M6_bronze = None
bat = None
bhoref = None
bac = None
bll = None
b = None
M4_silver = None
M4_bronze = None
docsr = None
H_bronze = None
H_silver = None

config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()
KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"
external_storage = f"ingest{lz_key}external{env_name}"

spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

spark.conf.set(f"fs.azure.account.auth.type.{checkpoint_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{checkpoint_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{checkpoint_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{checkpoint_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{checkpoint_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

spark.conf.set(f"fs.azure.account.auth.type.{raw_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{raw_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{raw_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{raw_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{raw_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

spark.conf.set(f"fs.azure.account.auth.type.{landing_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{landing_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{landing_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{landing_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{landing_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

spark.conf.set(f"fs.azure.account.auth.type.{external_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{external_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{external_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{external_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{external_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

bronze_path = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
silver_path = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
gold_path = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/{state_under_test}"

import json
json_location = dbutils.fs.ls(f"{gold_path}/")[-1]
latest_json_location = json_location.name

#Set Paths
try:
    json_path = f"{gold_path}/{latest_json_location}/JSON/"
    M1_silver = f"{silver_path}/silver_appealcase_detail"
    M1_bronze = f"{bronze_path}/bronze_appealcase_crep_rep_floc_cspon_cfs"
    M6_bronze = f"{bronze_path}/bronze_caseadjudicator_adjudicator"
    M3_bronze = f"{bronze_path}/bronze_status_htype_clist_list_ltype_court_lsitting_adj"
    M2_silver = f"{silver_path}/silver_caseapplicant_detail"
    M3_silver = f"{silver_path}/silver_status_detail"
    C = f"{silver_path}/silver_appealcategory_detail"
    bhc = f"{bronze_path}/bronze_hearing_centres"
except:
    print(f"Error during fetch: {str(e)}")

#Create and Load Dataframes
json_data = spark.read.format("json").load(json_path)
M1_silver = spark.read.format("delta").load(M1_silver)
M1_bronze = spark.read.format("delta").load(M1_bronze)
M6_bronze = spark.read.format("delta").load(M6_bronze)
M3_bronze = spark.read.format("delta").load(M3_bronze)
M2_silver = spark.read.format("delta").load(M2_silver)
M3_silver = spark.read.format("delta").load(M3_silver)
C = spark.read.format("delta").load(C)
bhc = spark.read.format("delta").load(bhc)

# Load data that isn't loading
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("appealReferenceNumber", StringType(), True),
    StructField("witness1InterpreterSignLanguage", StringType(), True),
    StructField("witness2InterpreterSignLanguage", StringType(), True),
    StructField("witness3InterpreterSignLanguage", StringType(), True),
    StructField("witness4InterpreterSignLanguage", StringType(), True),
    StructField("witness5InterpreterSignLanguage", StringType(), True),
    StructField("witness6InterpreterSignLanguage", StringType(), True),
    StructField("witness7InterpreterSignLanguage", StringType(), True),
    StructField("witness8InterpreterSignLanguage", StringType(), True),
    StructField("witness9InterpreterSignLanguage", StringType(), True),
    StructField("witness10InterpreterSignLanguage", StringType(), True),
    StructField("witness1InterpreterSpokenLanguage", StringType(), True),
    StructField("witness2InterpreterSpokenLanguage", StringType(), True),
    StructField("witness3InterpreterSpokenLanguage", StringType(), True),
    StructField("witness4InterpreterSpokenLanguage", StringType(), True),
    StructField("witness5InterpreterSpokenLanguage", StringType(), True),
    StructField("witness6InterpreterSpokenLanguage", StringType(), True),
    StructField("witness7InterpreterSpokenLanguage", StringType(), True),
    StructField("witness8InterpreterSpokenLanguage", StringType(), True),
    StructField("witness9InterpreterSpokenLanguage", StringType(), True),
    StructField("witness10InterpreterSpokenLanguage", StringType(), True)
])

print(f"Before - Number of Fields : {str(len(json_data.columns))}")
missing_fields_json = spark.read.schema(schema).json(json_path)
json_data = json_data.join(
    missing_fields_json,
    json_data["appealReferenceNumber"] == missing_fields_json["appealReferenceNumber"],
    "left"
).drop(missing_fields_json["appealReferenceNumber"])
print(f"After - Adding Missing Fields : Count : {str(len(json_data.columns))}")

# Load additional tables needed by chain runners
try: M2_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_appealcase_caseappellant_appellant") if M2_bronze is None else M2_bronze
except: pass
try: M3_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_status_htype_clist_list_ltype_court_lsitting_adj") if M3_bronze is None else M3_bronze
except: pass
try: bat = spark.read.format("delta").load(f"{bronze_path}/bronze_appealtype") if bat is None else bat
except: pass
try: bhoref = spark.read.format("delta").load(f"{bronze_path}/bronze_HORef_cleansing") if bhoref is None else bhoref
except: pass
try: H_silver = spark.read.format("delta").load(f"{silver_path}/silver_history_detail") if H_silver is None else H_silver
except: pass
try: H_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_history") if H_bronze is None else H_bronze
except: pass
try: M4_silver = spark.read.format("delta").load(f"{silver_path}/silver_transaction_detail") if M4_silver is None else M4_silver
except: pass
try: M4_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_appealcase_transaction_transactiontype") if M4_bronze is None else M4_bronze
except: pass
try: M6_bronze = spark.read.format("delta").load(f"{bronze_path}/bronze_history") if M6_bronze is None else M6_bronze
except: pass
try: bac = spark.read.format("delta").load(f"{bronze_path}/bronze_appealcategory") if bac is None else bac
except: pass
try: bll = spark.read.format("delta").load(f"{bronze_path}/bronze_listing_location") if bll is None else bll
except: pass
try: docsr = spark.read.format("delta").load(f"{bronze_path}/bronze_documentsreceived") if docsr is None else docsr
except: pass


In [0]:
import importlib
import Test_Functions.Ended_Tests as ended_tests
import Test_Functions.run_ended as run_ended

importlib.reload(ended_tests)
importlib.reload(run_ended)

In [0]:
import time as perf_time
perf_timings = []

from Test_Functions.paymentPending_appealType_caseData_hearingCentre_flagLabels_general import run as run_pp_chunk1
from Test_Functions.paymentPending_appellantDetails_legalRepDetails_sponsorDetails import run as run_pp_chunk2
from Test_Functions.paymentPending_partyIds_payment_remission_homeOffice_defaultMappings import run as run_pp_chunk3
from Test_Functions.paymentPending_Detained import run as run_pp_chunk4
from Test_Functions.test_helpers import classify_all
from Test_Functions.run_appealSubmitted import run_all_tests as run_as
from Test_Functions.run_awaitingRespEvidenceA import run_all_tests as run_areA
from Test_Functions.run_awaitingRespEvidenceB import run_all_tests as run_areB
from Test_Functions.run_caseUnderReview import run_all_tests as run_cur
from Test_Functions.run_reasonsForAppeal import run_all_tests as run_rfa
from Test_Functions.run_listing import run_all_tests as run_listing
from Test_Functions.run_prepareForHearing import run_all_tests as run_pfh
from Test_Functions.run_decision import run_all_tests as run_decision
from Test_Functions.run_decidedA import run_all_tests as run_decidedA
from Test_Functions.run_ftpaSubmittedA import run_all_tests as run_ftpaA
from Test_Functions.run_decidedB import run_all_tests as run_decidedB
from Test_Functions.run_ftpaSubmittedB import run_all_tests as run_ftpaB
from Test_Functions.run_ftpaDecided import run_all_tests as run_ftpaDecided
from Test_Functions.run_ended import run_all_tests as run_ended
from Test_Functions.run_remitted import run_all_tests as run_remitted

# print(f"Running 18 steps...")

# print(f"  Starting paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)...")
# t0 = perf_time.time()
# res = run_pp_chunk1(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, [], M4_silver, M4_bronze, M2_silver, H_silver, state_under_test)
# all_test_results = list(res)
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(res)})
# print(f"  [1/18] paymentPending Part 1 (appealType, caseData, hearingCentre, flagLabels, general)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)...")
# t0 = perf_time.time()
# res = run_pp_chunk2(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, [], M4_silver, M4_bronze, M2_silver, H_silver, state_under_test)
# all_test_results.extend(res)
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [2/18] paymentPending Part 2 (appellantDetails, legalRepDetails, sponsorDetails)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)...")
# t0 = perf_time.time()
# res = run_pp_chunk3(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, [], M4_silver, M4_bronze, M2_silver, H_silver, state_under_test)
# all_test_results.extend(res)
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [3/18] paymentPending Part 3 (partyIds, payment, remission, homeOffice, defaultMappings)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting paymentPending Part 4 (detained)...")
# t0 = perf_time.time()
# res = run_pp_chunk4(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, [], M4_silver, M4_bronze, M2_silver, H_silver, state_under_test)
# all_test_results.extend(res)
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "paymentPending Part 4 (detained)", "test_from_state": "paymentPending", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [4/18] paymentPending Part 4 (detained)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")


# print(f"  Starting appealSubmitted...")
# t0 = perf_time.time()
# all_test_results.extend(run_as(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bat, bhoref, external_storage, spark, []))
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "appealSubmitted", "test_from_state": "appealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [5/18] appealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting awaitingRespondentEvidence(a)...")
# t0 = perf_time.time()
# all_test_results.extend(run_areA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, []))
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "awaitingRespondentEvidence(a)", "test_from_state": "awaitingRespondentEvidence(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [6/18] awaitingRespondentEvidence(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting awaitingRespondentEvidence(b)...")
# t0 = perf_time.time()
# all_test_results.extend(run_areB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, []))
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "awaitingRespondentEvidence(b)", "test_from_state": "awaitingRespondentEvidence(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [7/18] awaitingRespondentEvidence(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting caseUnderReview...")
# t0 = perf_time.time()
# all_test_results.extend(run_cur(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, []))
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "caseUnderReview", "test_from_state": "caseUnderReview", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [8/18] caseUnderReview".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting reasonsForAppealSubmitted...")
# t0 = perf_time.time()
# all_test_results.extend(run_rfa(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, C, bhc, []))
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "reasonsForAppealSubmitted", "test_from_state": "reasonsForAppealSubmitted", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [9/18] reasonsForAppealSubmitted".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting listing...")
# t0 = perf_time.time()
# all_test_results.extend(run_listing(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bac, bll, b, []))
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "listing", "test_from_state": "listing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [10/18] listing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting prepareForHearing...")
# t0 = perf_time.time()
# all_test_results.extend(run_pfh(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, []))
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "prepareForHearing", "test_from_state": "prepareForHearing", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [11/18] prepareForHearing".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting decision...")
# t0 = perf_time.time()
# all_test_results.extend(run_decision(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, bll, spark, []))
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "decision", "test_from_state": "decision", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [12/18] decision".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting decided(a)...")
# t0 = perf_time.time()
# all_test_results.extend(run_decidedA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, []))
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "decided(a)", "test_from_state": "decided(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [13/18] decided(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting ftpaSubmitted(a)...")
# t0 = perf_time.time()
# all_test_results.extend(run_ftpaA(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, []))
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "ftpaSubmitted(a)", "test_from_state": "ftpaSubmitted(a)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [14/18] ftpaSubmitted(a)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting decided(b)...")
# t0 = perf_time.time()
# all_test_results.extend(run_decidedB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M6_bronze, C, bhc, []))
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "decided(b)", "test_from_state": "decided(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [15/18] decided(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting ftpaSubmitted(b)...")
# t0 = perf_time.time()
# all_test_results.extend(run_ftpaB(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, []))
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "ftpaSubmitted(b)", "test_from_state": "ftpaSubmitted(b)", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [16/18] ftpaSubmitted(b)".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

# print(f"  Starting ftpaDecided...")
# t0 = perf_time.time()
# all_test_results.extend(run_ftpaDecided(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, C, bhc, []))
# dur = perf_time.time() - t0
# perf_timings.append({"test_name": "ftpaDecided", "test_from_state": "ftpaDecided", "elapsed_seconds": dur, "result_count": len(all_test_results)})
# print(f"  [17/18] ftpaDecided".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results")

print(f"  Starting ended...")
t0 = perf_time.time()
all_test_results.extend(run_ended(json_data, M1_bronze, M1_silver, M2_bronze, M3_bronze, M3_silver, M6_bronze, bac, C, bhc, fields_to_exclude))
dur = perf_time.time() - t0
perf_timings.append({"test_name": "ended", "test_from_state": "ended", "elapsed_seconds": dur, "result_count": len(all_test_results)})
print(f"  [18/18] ended".ljust(75) + f"{int(dur//3600)}h {int(dur%3600//60)}m {int(dur%60)}s ({dur:.1f}s)  {len(all_test_results)} results -- all complete")
display(all_test_results)

print(f"  Finished running all states.")
print(f"Total tests collected: {len(all_test_results)}")

In [0]:
import importlib
import Test_Functions.Ended_Tests as ended_tests
import Test_Functions.run_ended as run_ended

importlib.reload(ended_tests)
importlib.reload(run_ended)

In [0]:
from pyspark.sql import functions as F

# Check if Appellant Name exists but Prefix is empty
display(test_df.select(
    "appealReferenceNumber", 
    "Appellant_Name", 
    "bundleFileNamePrefix"
).filter(F.col("bundleFileNamePrefix").isNull()))

In [0]:
from pyspark.sql import functions as F

target_case = "PA/04670/2024"

print(f"Full Status History in ARIA for {target_case}:")
display(M3_silver.filter(F.col("CaseNo") == target_case)
        .select("CaseNo", "StatusId", "CaseStatus", "Outcome")
        .orderBy(F.desc("StatusId")))

print(f"Current JSON Mapping for {target_case}:")
display(json_data.filter(F.col("appealReferenceNumber") == target_case)
        .select("appealReferenceNumber", "endAppealOutcome", "endAppealOutcomeReason"))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# 1. Identify the legal final outcome from Bronze (Priority 1)
substantive_statuses = [37, 38, 26, 39, 10]
window_spec = Window.partitionBy("CaseNo").orderBy(F.desc("StatusId"))

aria_substantive = m3_bronze.filter(F.col("CaseStatus").isin(substantive_statuses)) \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") == 1) \
    .select(
        F.col("CaseNo").alias("ARIA_CaseNo"),
        F.col("CaseStatus").alias("ARIA_Legal_Status"),
        F.col("Outcome").alias("ARIA_Legal_Outcome")
    )

# 2. Join your current test results (test_df) with the Substantive ARIA data
comparison_report = test_df.join(aria_substantive, test_df.appealReferenceNumber == aria_substantive.ARIA_CaseNo, "inner") \
    .select(
        F.col("appealReferenceNumber").alias("Case_Number"),
        F.col("endAppealOutcome").alias("JSON_Outcome_Result"),        # The Buggy Result
        F.col("ARIA_Legal_Status").alias("Actual_Legal_Status"),       # e.g. 37
        F.col("ARIA_Legal_Outcome").alias("Actual_Legal_Outcome"),     # e.g. 25
        F.col("endAppealOutcomeReason").alias("JSON_Full_Reason")      # The Reason String
    )

# 3. Filter for cases where JSON says 'Struck out' but ARIA says it was a Hearing (Status 37/38)
conflicts = comparison_report.filter(
    (F.col("JSON_Outcome_Result") == "Struck out") & 
    (F.col("Actual_Legal_Status").isin([37, 38]))
)

print("EVIDENCE REPORT: Cases where a Hearing was overwritten by a Fee/Admin Closure")
display(conflicts)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# 1. Prepare the Substantive ARIA data and give it a clear alias "aria"
substantive_statuses = [37, 38, 26, 39, 10]
window_spec = Window.partitionBy("CaseNo").orderBy(F.desc("StatusId"))

aria_df = M3_bronze.filter(F.col("CaseStatus").isin(substantive_statuses)) \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") == 1) \
    .select("CaseNo", "CaseStatus", "Outcome") \
    .alias("aria")  # <--- Unique name for this dataset

# 2. Give test_df a clear alias "json"
json_df = test_df.alias("json") # <--- Unique name for your test results

# 3. Join using the unique aliases to specify exactly which column to pick
comparison_report = json_df.join(
    aria_df, 
    F.col("json.appealReferenceNumber") == F.col("aria.CaseNo"), 
    "inner"
).select(
    F.col("json.appealReferenceNumber"),
    F.col("json.endAppealOutcome"),
    F.col("aria.CaseStatus"),  # Now Spark knows exactly which one this is
    F.col("aria.Outcome"),     # Now Spark knows exactly which one this is
    F.col("json.endAppealOutcomeReason")
)

# 4. Filter for the "Conflict" (JSON says Struck Out, but ARIA says Hearing)
conflicts = comparison_report.filter(
    (F.col("json.endAppealOutcome") == "Struck out") & 
    (F.col("aria.CaseStatus").isin([37, 38]))
)

print("EVIDENCE REPORT: Cases where a Hearing was overwritten by a Fee/Admin Closure")
display(conflicts)

In [0]:
from pyspark.sql import functions as F

# Check the specific failing Case
target_case = "PA/04670/2024"

# If Status 52 is included in your criteria, it's 'winning' because it has the highest ID
criteria_statuses = [10, 46, 26, 37, 38, 39, 51, 52, 36]

print(f"Potential 'End Appeal' records for {target_case}:")
display(M3_silver.filter(
    (F.col("CaseNo") == target_case) & 
    (F.col("CaseStatus").isin(criteria_statuses))
).select("StatusId", "CaseStatus", "Outcome").orderBy(F.desc("StatusId")))

In [0]:
from pyspark.sql import functions as F

# 1. Identify cases that have BOTH a Hearing Outcome and a later Fee Outcome
substantive_outcomes = [37, 38, 26]
admin_outcomes = [51, 52]

# Get the max ID for substantive vs admin per case
comparison_df = M3_silver.groupBy("CaseNo").agg(
    F.max(F.when(F.col("CaseStatus").isin(substantive_outcomes), F.col("StatusId"))).alias("Max_Substantive_ID"),
    F.max(F.when(F.col("CaseStatus").isin(admin_outcomes), F.col("StatusId"))).alias("Max_Admin_ID")
)

# Find cases where Admin happened AFTER Substantive
conflicts = comparison_df.filter(
    (F.col("Max_Admin_ID") > F.col("Max_Substantive_ID")) & 
    (F.col("Max_Substantive_ID").isNotNull())
)

print(f"Detected {conflicts.count()} cases where an administrative closure is likely overwriting a substantive outcome.")
display(conflicts.limit(10))

In [0]:
# Check if there are ANY records for these statuses before the join
display(M3_silver.filter(col("CaseStatus").isin(26, 37, 38))
        .groupBy("CaseStatus", "Outcome")
        .count())

In [0]:
from pyspark.sql import functions as F

defect_df = investigation_df.filter(
    (F.col("m1.LanguageId").isNotNull()) & 
    (F.col("m1.LanguageId") != 0) & 
    (F.col("json.appellantInterpreterSpokenLanguage").isNull())
)

print(f"DEFECTS FOUND: {defect_df.count()} rows missing data.")

display(defect_df.select(
    F.col("appealReferenceNumber").alias("CaseNo"),
    F.col("LanguageId").alias("Missing_ARIA_ID"),
    F.lit("Object is NULL in JSON").alias("Failure_Type")
))

In [0]:
from pyspark.sql.functions import col, lit, when, array, array_union, array_sort, create_map, coalesce, lower
display(json_data.filter(col("appealReferenceNumber") == "PA/03185/2020")
        .select("appealReferenceNumber", "appellantInterpreterSpokenLanguage"))

# isEvidenceFromOutsideUkOoc - failure investigation

In [0]:
# Scenario: Group 3/4 and Category 38, but value is NOT "Yes"
investigation_test1 = test_df.filter(
    (F.col("EndedGroup").isin(3, 4)) & 
    (F.col("CategoryId") == 38) & 
    (F.col("isEvidenceFromOutsideUkOoc") != "Yes")
)

print(f"Investigating {investigation_test1.count()} Value Mismatches:")
display(investigation_test1.select(
    "appealReferenceNumber", 
    "EndedGroup", 
    "CategoryId", 
    "Sponsor_Name", 
    "isEvidenceFromOutsideUkOoc"
))

In [0]:
# Scenario: Group 3/4 and Category is NOT 38, but field is NOT Null
investigation_test2 = test_df.filter(
    (F.col("EndedGroup").isin(3, 4)) & 
    (F.col("CategoryId") != 38) & 
    (F.col("isEvidenceFromOutsideUkOoc").isNotNull())
)

print(f"Investigating {investigation_test2.count()} Category Leakage rows:")
display(investigation_test2.select(
    "appealReferenceNumber", 
    "EndedGroup", 
    "CategoryId", 
    "Sponsor_Name", 
    "isEvidenceFromOutsideUkOoc"
))

# isEvidenceFromOutsideUkInCountry - failure investigation

In [0]:
# Filter for Category 37 records that failed to return "Yes"
investigation_test1 = test_df.filter(
    (F.col("EndedGroup").isin(3, 4)) & 
    (F.col("CategoryId") == 37) & 
    (F.col("isEvidenceFromOutsideUkInCountry") != "Yes")
)

print(f"Investigating {investigation_test1.count()} Category 37 Value Failures:")
display(investigation_test1.select(
    "appealReferenceNumber", 
    "EndedGroup", 
    "CategoryId", 
    "Sponsor_Name", 
    "isEvidenceFromOutsideUkInCountry"
))

In [0]:
# Filter for Group 3/4 where Category is NOT 37, but field is NOT Null
investigation_test2 = test_df.filter(
    (F.col("EndedGroup").isin(3, 4)) & 
    (F.col("CategoryId") != 37) & 
    (F.col("isEvidenceFromOutsideUkInCountry").isNotNull())
)

print(f"Investigating {investigation_test2.count()} Category Leakage Failures:")
display(investigation_test2.select(
    "appealReferenceNumber", 
    "EndedGroup", 
    "CategoryId", 
    "isEvidenceFromOutsideUkInCountry"
))

In [0]:
########################
# PROCESS RESULTS
########################
from collections import defaultdict
from Test_Functions.report_helpers import (
    count_by_status,
    create_run, create_results, create_perf_results,
    build_report_folder,
    write_csv, write_xlsx, write_html,
)

# ---- 1. COUNTS (visible, shared with helpers) ----
counts = count_by_status(all_test_results)

# ---- 2. CONSOLE SUMMARY (visible, easy to tweak) ----
grouped = defaultdict(list)
for r in all_test_results:
    grouped[r.test_from_state].append(r)

lines = []
lines.append("=" * 80)
lines.append(f"TEST EXECUTION REPORT -- {state_under_test}")
lines.append("-" * 80)
lines.append(f"Total tests : {counts['TOTAL']}")
lines.append(f"Passed      : {counts['PASS']}")
lines.append(f"Failed      : {counts['FAIL']}")
lines.append(f"No Data     : {counts['NO_DATA']}")
lines.append(f"Error       : {counts['ERROR']}")
lines.append("-" * 80)

for state in sorted(grouped):
    c = count_by_status(grouped[state])
    lines.append(f"\nSTATE: {state}")
    lines.append(f"Tests: {c['TOTAL']} | Passed: {c['PASS']} | Failed: {c['FAIL']} | No Data: {c['NO_DATA']} | Error: {c['ERROR']}")
    lines.append("-" * 60)
print("\n".join(lines))
# ---- 2b. PERFORMANCE TIMINGS ----
if perf_timings:
    lines2 = []
    lines2.append("")
    lines2.append("=" * 80)
    lines2.append("PERFORMANCE TIMINGS")
    lines2.append("-" * 80)
    total_secs = 0.0
    for pt in perf_timings:
        total_secs += pt["elapsed_seconds"]
        lines2.append(f"  {pt['test_name']:65s} {int(pt['elapsed_seconds']//3600)}h {int(pt['elapsed_seconds']%3600//60)}m {int(pt['elapsed_seconds']%60)}s ({pt['elapsed_seconds']:.1f}s)  {pt['result_count']} results")
    lines2.append("-" * 80)
    lines2.append(f"  {'TOTAL':65s} {int(total_secs//3600)}h {int(total_secs%3600//60)}m {int(total_secs%60)}s ({total_secs:.1f}s)")
    lines2.append("=" * 80)
    print("\n".join(lines2))


# ---- 3. IF THIS IS THE TOP STATE: WRITE FILES + TABLES ----
if state_under_test == this_notebook_state:
    # Build ONE folder for this run so all 3 files land together
    folder, timestamp, _ = build_report_folder(test_results_path, state_under_test)
    print(f"Reports folder: {folder}")

    try:
        csv_path = write_csv(all_test_results, state_under_test, folder, timestamp)
        print(f"CSV  : {csv_path}")
    except Exception as e:
        print(f"CSV  write failed: {e}")

    try:
        xlsx_path = write_xlsx(all_test_results, state_under_test, folder, timestamp)
        print(f"XLSX : {xlsx_path}")
    except Exception as e:
        print(f"XLSX write failed: {e}")

    try:
        html_path = write_html(all_test_results, state_under_test, folder, timestamp, counts=counts)
        print(f"HTML : {html_path}")
    except Exception as e:
        print(f"HTML write failed: {e}")

    # Spark tables
    run_status = "PASS" if counts["FAIL"] == 0 and counts["PASS"] >= 1 else "FAIL"
    try:
        run_id = create_run(
            spark,
            run_user=run_user,
            run_start_datetime=run_start_datetime,
            run_by_automation_name=run_by_automation_name,
            run_tag=run_tag,
            run_status=run_status,
            state_under_test=state_under_test,
        )
        print(f"Created run_id={run_id}")
    except Exception as e:
        print(f"create_run FAILED: {e}")
        run_id = None

    if run_id:
        try:
            n = create_results(spark, run_id, all_test_results)
            print(f"Saved {n} result rows")
        except Exception as e:
            print(f"create_results FAILED: {e}")

        try:
            np = create_perf_results(spark, run_id, perf_timings, state_under_test)
            print(f"Saved {np} perf rows")
        except Exception as e:
            print(f"create_perf_results FAILED: {e}")

else:
    # Chain child — return data to parent
    import json
    from dataclasses import asdict
    all_test_results_string = json.dumps([asdict(r) for r in all_test_results])
    print(f"Exiting Notebook : {this_notebook_state} and returning Data to Parent notebook : {state_under_test}")
    dbutils.notebook.exit(all_test_results_string)